---

## Capstone · Part 1 — Count and compare the closures

Everything above was practice on the diabetes data. Now we turn to the project that runs through both courses: **rural hospital closures**. You cleaned and analyzed this data in the pandas course; here you begin to *show* it — and this section uses only the skills from Notebooks 2 and 3.

**Driving question.** Which states and payment models bore the brunt of rural hospital closures — and were the hospitals that closed big or small?

We start from the cleaned data. The cell below reproduces, in one step, the tidy-up you did in the pandas capstone — dropping the saved index, fixing the mistyped closure year, standardizing `closure_type`, filling the few missing bed counts, removing bed-count outliers with the 1.5 × IQR rule, and dropping rows with no state — so we begin analysis-ready and spend our time on the charts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"

In [ ]:
closures = pd.read_csv(f"{BASE_URL}/rural_hospital_closures.csv")

closures = closures.drop(columns="Unnamed: 0")
closures["closure_year"] = closures["closure_year"].replace(1019, 2019)
closures["closure_type"] = (closures["closure_type"].str.strip().str.capitalize()
                            .replace({"Complet": "Complete", "Convertd": "Converted"}))
closures["beds"] = closures["beds"].fillna(closures["beds"].median())
q1, q3 = closures["beds"].quantile([0.25, 0.75])
iqr = q3 - q1
closures = closures[closures["beds"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]
closures = closures.dropna(subset=["state"])
closures.shape

### Task 1 — Which states lost the most?

Make a horizontal count plot of the **top 10 states** by number of closures, ordered most-to-least, fully labeled.

In [ ]:
# Your work here
top_states = closures["state"].value_counts().head(10).index # use state name as the index
top_states

In [ ]:
closures["state"].value_counts().head(10).reset_index() 
# go back to using numbers as the index, state becomes a column

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=closures, y="state", order=top_states, color="steelblue", ax=ax)
#sns.countplot(data=closures, y="state", order=top_states, color="steelblue", edgecolor="black", ax=ax)
#sns.countplot(data=closures, y="state", order=top_states, color="steelblue", width=0.3, edgecolor="black", ax=ax)

ax.set_title("Rural hospital closures by state (top 10)")
ax.set_xlabel("Closures")
ax.set_ylabel("State")
fig.tight_layout()
plt.show()

In [ ]:

# 
# for countplot - stat is limited to count-based options (count, percent, proportion, probability)
# 
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(
    data=closures,
    y="state",
    order=top_states,
    color="steelblue",
    edgecolor="black",      # outline each bar
    stat="percent",         # show share of total instead of raw count
    ax=ax,
)
ax.set_title("Rural hospital closures by state (top 10)")
ax.set_xlabel("Percent of all closures")   # updated to match stat="percent"
ax.set_ylabel("State")
fig.tight_layout()
plt.show()

<details>
<summary><b>Solution</b></summary>

```python
top_states = closures["state"].value_counts().head(10).index

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=closures, y="state", order=top_states, color="steelblue", ax=ax)
ax.set_title("Rural hospital closures by state (top 10)")
ax.set_xlabel("Closures")
ax.set_ylabel("State")
fig.tight_layout()
plt.show()
# Texas, Oklahoma, and Tennessee lead; closures cluster heavily in the rural South.
```

**Why this works.** `value_counts().head(10).index` is both the shortlist and the display order, and a `y=` count plot draws it as a clean ranking — the horizontal-bar pattern from Section 5, now on real stakes.

</details>

### Task 2 — Big hospitals or small?

Show the distribution of `beds` for the hospitals that closed: a histogram on top, a boxplot beneath, sharing the x-axis. In a comment, give the median bed count.

In [ ]:
# Your work here
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

sns.histplot(closures["beds"], binwidth=5, color="steelblue", ax=axes[0])
axes[0].set_title("Bed counts of closed hospitals")
axes[0].set_ylabel("Hospitals")

sns.boxplot(x=closures["beds"], color="steelblue", ax=axes[1])
axes[1].set_xlabel("Beds")

fig.tight_layout()
plt.show()
# Small: median is 25 beds, and the whole distribution sits under about 55.

<details>
<summary><b>Solution</b></summary>

```python
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

sns.histplot(closures["beds"], binwidth=5, color="steelblue", ax=axes[0])
axes[0].set_title("Bed counts of closed hospitals")
axes[0].set_ylabel("Hospitals")

sns.boxplot(x=closures["beds"], color="steelblue", ax=axes[1])
axes[1].set_xlabel("Beds")

fig.tight_layout()
plt.show()
# Small: median is 25 beds, and the whole distribution sits under about 55.
```

**Why this works.** This is the histogram-over-boxplot pattern from Notebook 2, pointed at the capstone data. The answer to the driving question is right there in the shape: the hospitals that close are small community hospitals, not large regional ones.

</details>

### Task 3 — Do some payment models lose bigger hospitals?

Plot the **mean `beds` by `payment_type`**, ordered by height, with an honest zero baseline. (The `payment_types.csv` file in the repo documents what each code means.)

In [ ]:
# Your work here
order = closures.groupby("payment_type")["beds"].mean().sort_values().index
order

**Step 1** — closures.groupby("payment_type")
Splits the DataFrame's rows into groups, one per unique value of payment_type (Medicare, Medicaid, Private, …). Nothing is computed yet; this just produces a DataFrameGroupBy object that knows which rows belong to which group.

**Step 2** — ["beds"]
Narrows the grouping to a single column, beds. Now it's a SeriesGroupBy — the grouping plus the one numeric column you want to summarize. This is what lets the next step aggregate beds rather than every column.

**Step 3** — .mean()
Collapses each group to a single number: the average beds within that group. The result is a Series whose index is payment_type and whose values are the group means (e.g. Medicaid 22.5, Medicare 55.0, Private 90.0). By default mean() skips NaN values.

**Step 4** — .sort_values()
Sorts that Series by its values (the means), ascending by default. So the payment types get reordered from lowest mean beds to highest. The index travels with the values, so the labels are now in mean-sorted order. (Add ascending=False to flip it; any groups with NaN means land at the end.)

**Step 5** — .index
Discards the numeric values and keeps just the labels — the payment_type names, now in ascending-mean order. The result is a pandas Index object, assigned to order.

In [ ]:


fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=closures, x="payment_type", y="beds", order=order, color="steelblue", ax=ax)
ax.set_title("Mean beds by Medicare payment type")
ax.set_xlabel("Payment type")
ax.set_ylabel("Mean beds")
fig.tight_layout()
plt.show()

<details>
<summary><b>Solution</b></summary>

```python
order = closures.groupby("payment_type")["beds"].mean().sort_values().index

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=closures, x="payment_type", y="beds", order=order, color="steelblue", ax=ax)
ax.set_title("Mean beds by Medicare payment type")
ax.set_xlabel("Payment type")
ax.set_ylabel("Mean beds")
fig.tight_layout()
plt.show()
# RRC hospitals that closed are the largest on average (~35 beds); REH the smallest (~19).
```

**Why this works.** It is the ordered aggregate bar from Section 3, on `payment_type`. Because `payment_type` is unordered, sorting by the mean is the right call, and the zero baseline keeps the modest differences honestly modest.

</details>

## Error bars

errorbar — what the bar represents. Options:

- ("ci", 95) — 95% confidence interval (the default)
- ("ci", 99) — a different confidence level (wider)
- "sd" or ("sd", 2) — standard deviation (n multiples). Notice these are much taller — SD describes spread of the data, not uncertainty of the mean, so it doesn't shrink with sample size
- ("se", 1) — standard error (n multiples); the smallest here
- ("pi", 95) — a percentile interval
- None — removes the bars entirely (bottom-right panel)

- capsize — adds the little horizontal caps at the ends (a fraction of bar width, e.g. 0.15). By default there are no caps.

- err_kws — a dict of matplotlib line properties to style the bars themselves: color, linewidth, alpha, etc. This is the current way to restyle them; the old errcolor/errwidth arguments are deprecated in 0.13.

In [ ]:
# Error bars

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=closures, x="payment_type", y="beds", order=order,
    color="steelblue", ax=ax,
    errorbar=("ci", 95),                       # or "sd", ("se",1), None, ...
    capsize=0.15,                              # add end caps
    err_kws={"color": "0.2", "linewidth": 1.5} # style the whiskers
)
ax.set_title("Mean beds by Medicare payment type")
ax.set_xlabel("Payment type")
ax.set_ylabel("Mean beds")
fig.tight_layout()
plt.show()

### One more look — does closure *type* relate to size?

Before the wrap, one comparison worth drawing honestly: do hospitals that close *completely* differ in size from those that *convert* to another kind of facility?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=closures, x="closure_type", y="beds", errorbar=None, color="steelblue", ax=ax)
ax.set_title("Mean beds by closure type (honest baseline)")
ax.set_xlabel("Closure type")
ax.set_ylabel("Mean beds")
fig.tight_layout()
plt.show()

### The insight — and one surprise

**Insight.** Rural closures are not spread evenly. A handful of southern states — Texas, Oklahoma, Tennessee, Alabama — carry a large share, and the hospitals that close are overwhelmingly small, with a median of 25 beds. Geography and size, not payment model, tell most of the story.

**One surprise.** Whether a hospital *closed completely* or *converted* to another kind of facility has almost nothing to do with its size: the mean bed counts are 29.8 and 28.7 — about one bed apart. On the zero baseline above, the two bars look identical, which is the honest reading. It would take exactly the truncated axis you saw in Section 6 to inflate that one-bed gap into a finding — a standing reminder to distrust any version of this chart whose y-axis does not start at zero.

**Next:** Session 2 picks these closures back up — alongside new relationships and time series — and carries them all the way to a presentation-ready figure.